# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print("Total revenue:", round(total_revenue, 2))
print("Total units:", total_units)


Total revenue: 8520.0
Total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [4]:
# TODO
rev_by_cat = (
    df.groupby('category')['revenue']
      .sum()
      .sort_values(ascending=False)
      .to_frame('revenue')
)

total_rev = df['revenue'].sum()
rev_by_cat['share_pct'] = (rev_by_cat['revenue'] / total_rev * 100).round(2)

rev_by_cat

,revenue,share_pct
category,,
Food,4293.0,50.39
Merch,1771.5,20.79
Drink,1554.0,18.24
RainGear,901.5,10.58


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [6]:
# TODO
vendor_stats = (
    df.groupby('vendor_id')
      .agg(avg_revenue=('revenue','mean'),
           order_count=('revenue','size'))
      .sort_values('avg_revenue', ascending=False)
)

vendor_stats

,avg_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [7]:
# TODO
total_rev = df['revenue'].sum()
merch_rev = df.loc[df['category'] == 'Merch', 'revenue'].sum()

merch_share = round(merch_rev / total_rev * 100, 1)
print("Merch revenue share:", merch_share, "%")


Merch revenue share: 20.8 %


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [9]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
merged = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

print("Row count unchanged:", len(merged) == len(df))
print("Revenue unchanged:", merged['revenue'].sum() == df['revenue'].sum())

unmatched = merged.loc[merged['vendor_name'].isna(), 'vendor_id'].unique()
unmatched

Row count unchanged: True
Revenue unchanged: True


array(['V-18'], dtype=object)

**The unmatched vendor, and what I did about it:** _..._ The unmatched vendor is V-18. Since it ahs real orders in the dataset but, no entry in the lookup table. I left its vendor_name as NaN and would ask the client team to apply the missing name before prodcing a final report

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [10]:
# TODO
pivot = pd.pivot_table(
    df,
    values='revenue',
    index='vendor_id',
    columns='category',
    aggfunc='sum',
    fill_value=0
)

pivot['Total'] = pivot.sum(axis=1)

pivot.loc['Total'] = pivot.sum(axis=0)

pivot

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [11]:
by_category = rev_by_cat
joined = merged

assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'

print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**a)** Looking at the numbers, the crowd clearly didn't spend evenly across categories. Total revenue for the night was a little over eight to nine thousand dollars, but almost half of that came from Food, while Merch barely cracked ten to fifteen percent. If vendors want to be better prepared next game, the Food teams should bring more inventory and staff because that's where the demand actually is. Merch, on the other hand, could scale back a bit or rethink their product mix. The vendor average also Varied a lot. For example, one vendor's average order was nearly twice another's. This suggests some vendors might benefit from adjusting pricing or offering bundles to boost their per-order revenue.

**b)** The shakiest result is definitely Q3, the "highest average order revenue" question. Averages only make sense when the group sizes are comparable, and here they aren't. A vendor with twelve orders will have a jumpy, unreliable average compared to One with 200 orders, so the ranking can exaggerate performance. On top of that, V-18 didn't have a name in the lookup table, which means any interpretation involving vendor identity is incomplete. Between the tiny sample size and the missing metadata, Q3 is the one I'd trust the least.